In [1]:
import pandas as pd

URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"
COLUMNS = ["age", "workclass", "fnlwgt", "education", "education_num",
           "marital_status", "occupation", "relationship", "race", "sex",
           "capital_gain", "capital_loss", "hours_per_week", "native_country", "income"]

df = pd.read_csv(URL, header=None, names=COLUMNS, skipinitialspace=True)
df.shape


(32561, 15)

In [2]:
df.head()

,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


In [3]:
df.dtypes

age               int64
workclass           str
fnlwgt            int64
education           str
education_num     int64
marital_status      str
occupation          str
relationship        str
race                str
sex                 str
capital_gain      int64
capital_loss      int64
hours_per_week    int64
native_country      str
income              str
dtype: object

## 1. Data-quality audit
### 1a. Missingness

In [4]:
df.isna().sum()

age               0
workclass         0
fnlwgt            0
education         0
education_num     0
marital_status    0
occupation        0
relationship      0
race              0
sex               0
capital_gain      0
capital_loss      0
hours_per_week    0
native_country    0
income            0
dtype: int64

In [5]:
missing_q = (df == "?").sum()
missing_q[missing_q > 0]

workclass         1836
occupation        1843
native_country     583
dtype: int64

In [6]:
pd.crosstab(df["workclass"] == "?", df["occupation"] == "?")

occupation,False,True
workclass,,
False,30718,7
True,0,1836


In [7]:
has_missing = (df[["workclass", "occupation", "native_country"]] == "?").any(axis=1)
print(has_missing.sum(), "rows have at least one '?'")
df.groupby(has_missing)["income"].value_counts(normalize=True).unstack()

2399 rows have at least one '?'


income,<=50K,>50K
False,0.751078,0.248922
True,0.861192,0.138808


In [8]:
df.loc[(df["occupation"] == "?") & (df["workclass"] != "?"), "workclass"].value_counts()

workclass
Never-worked    7
Name: count, dtype: int64

**Finding 1 – Hidden, non-random missingness.**
`df.isna()` reports 0 missing values, but the raw file encodes missing as `"?"`:
workclass 1,836 · occupation 1,843 · native_country 583.
- workclass and occupation are missing together in the same 1,836 rows; the 7 extra
  occupation gaps are all `Never-worked` (occupation not applicable, not truly missing).
- Rows with any `?` earn >50K at 13.9% vs 24.9% for complete rows → missingness is
  related to the target (not missing at random).
- Decision: do NOT drop these rows (that would remove a lower-income group and bias the
  data). Replace `"?"` with an explicit `"Missing"` category so the model can use it.

### 1b. Class imbalance

In [9]:
df["income"].value_counts(normalize=True)

income
<=50K    0.75919
>50K     0.24081
Name: proportion, dtype: float64

### 1c. Protected attributes & sampling skew

In [10]:
for col in ["sex", "race"]:
    summary = df.groupby(col)["income"].agg(
        share_of_data=lambda s: len(s) / len(df),
        pct_over_50k=lambda s: (s == ">50K").mean(),
    )
    display(summary.round(3))

,share_of_data,pct_over_50k
sex,,
Female,0.331,0.109
Male,0.669,0.306


,share_of_data,pct_over_50k
race,,
Amer-Indian-Eskimo,0.010,0.116
Asian-Pac-Islander,0.032,0.266
Black,0.096,0.124
Other,0.008,0.092
White,0.854,0.256


**Finding 2 – Class imbalance.** Only 24.1% of rows are `>50K`. A model that always predicts
`<=50K` gets 75.9% accuracy, so accuracy alone is misleading. → Use a stratified split and
report ROC-AUC and F1 alongside accuracy.

**Finding 3 – Sampling skew and outcome gaps on protected attributes.**
- Sex: 66.9% Male / 33.1% Female (population is ~50/50). >50K rate: Male 30.6% vs Female 10.9% (~2.8×).
- Race: 85.4% White; Amer-Indian-Eskimo (1.0%) and Other (0.8%) have ~300 rows each.
  >50K rate ranges from 9.2% (Other) to 26.6% (Asian-Pac-Islander).
- Risk: a model will learn these historical gaps; per-group metrics for small race groups
  will be unreliable (~55 test rows for "Other").

### 1d. Leakage, redundancy & proxy checks

In [11]:
df.groupby("education")["education_num"].unique()

education
10th             [6]
11th             [7]
12th             [8]
1st-4th          [2]
5th-6th          [3]
7th-8th          [4]
9th              [5]
Assoc-acdm      [12]
Assoc-voc       [11]
Bachelors       [13]
Doctorate       [16]
HS-grad          [9]
Masters         [14]
Preschool        [1]
Prof-school     [15]
Some-college    [10]
Name: education_num, dtype: object

In [12]:
df.groupby("income")["fnlwgt"].median()

income
<=50K    179465.0
>50K     176101.0
Name: fnlwgt, dtype: float64

In [13]:
pd.crosstab(df["relationship"], df["sex"])

sex,Female,Male
relationship,,
Husband,1,13192
Not-in-family,3875,4430
Other-relative,430,551
Own-child,2245,2823
Unmarried,2654,792
Wife,1566,2


**Finding 4 – Redundant, non-informative and proxy columns.**
- `education` ↔ `education_num` are a perfect 1-to-1 mapping (e.g. Bachelors → 13).
  → Keep only `education_num` (already ordinal); drop `education`.
- `fnlwgt` is a census sampling weight, not a personal attribute; median is 179,465 (<=50K)
  vs 176,101 (>50K), ~2% difference → no signal. → Drop.
- `relationship` is a near-perfect proxy for `sex`: Husband = 13,192 Male / 1 Female,
  Wife = 1,566 Female / 2 Male (the 3 mismatches are likely entry errors). These two
  values cover 45% of rows, so removing `sex` alone does NOT stop a model from using it.

## 2. Preparation & train/test split
**Decision on protected attributes:** `sex` and `race` are excluded from model features and
kept aside only to audit the model's behaviour per group. This is not a full fix –
`relationship` still leaks sex (Finding 4) – so per-group results will be checked later.

In [14]:
clean = df.replace("?", "Missing")
clean = clean.drop(columns=["education", "fnlwgt"])

y = (clean["income"] == ">50K").astype(int)
protected = clean[["sex", "race"]]
X = clean.drop(columns=["income", "sex", "race"])
X.shape

(32561, 10)

In [15]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test, prot_train, prot_test = train_test_split(
    X, y, protected, test_size=0.2, stratify=y, random_state=42
)
print(X_train.shape, X_test.shape)
print(y_train.mean().round(3), y_test.mean().round(3))

(26048, 10) (6513, 10)
0.241 0.241


## 3. Baseline model (before feature engineering)

In [16]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

def make_model(X):
    numeric = X.select_dtypes("number").columns
    categorical = X.select_dtypes(exclude="number").columns
    preprocess = ColumnTransformer([
        ("num", StandardScaler(), numeric),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical),
    ])
    return Pipeline([("prep", preprocess), ("model", LogisticRegression(max_iter=1000))])

In [17]:
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

def evaluate(model, X_test, y_test):
    pred = model.predict(X_test)
    prob = model.predict_proba(X_test)[:, 1]
    return {
        "accuracy": accuracy_score(y_test, pred),
        "f1": f1_score(y_test, pred),
        "roc_auc": roc_auc_score(y_test, prob),
    }

In [18]:
baseline = make_model(X_train).fit(X_train, y_train)
baseline_scores = evaluate(baseline, X_test, y_test)
baseline_scores

{'accuracy': 0.8547520343927529,
 'f1': 0.670383275261324,
 'roc_auc': 0.9082682466313118}

## 4. Feature engineering (hypotheses first)

**F1 `log_capital_gain` = log(1 + capital_gain).**
Hypothesis: having investment income signals wealth, and what matters is its *order of
magnitude*, not the raw dollar amount. Raw capital_gain is extremely skewed (most rows are 0,
a few are 99,999), so after scaling the few huge values dominate and the model treats $3,000
and $0 as nearly the same. The log compresses that tail so every level of gain counts.

**F2 `age_group` = age cut into bands (<25, 25-34, 35-44, 45-54, 55-64, 65+).**
Hypothesis: income rises through the career, peaks around 45-54, then falls toward
retirement (an inverted U). A logistic regression can only learn "older = more" or
"older = less" from raw age, so it cannot represent the peak. Bands let each life stage
have its own effect.

**F3 `edu_x_hours` = education_num × hours_per_week.**
Hypothesis: education and hours worked multiply rather than add. A degree pays off mainly
when combined with full-time work (a part-time PhD or a full-time early school-leaver both
tend to earn less). A linear model cannot learn that interaction on its own.


In [19]:
import numpy as np

def add_features(X):
    X = X.copy()
    X["log_capital_gain"] = np.log1p(X["capital_gain"])
    X["age_group"] = pd.cut(
        X["age"], bins=[0, 25, 35, 45, 55, 65, 100], right=False,
        labels=["<25", "25-34", "35-44", "45-54", "55-64", "65+"],
    ).astype(str)
    X["edu_x_hours"] = X["education_num"] * X["hours_per_week"]
    return X

X_train_fe = add_features(X_train)
X_test_fe = add_features(X_test)
X_train_fe.shape

(26048, 13)

In [20]:
check = X_train_fe.assign(over_50k=y_train)
display(check.groupby("age_group")["over_50k"].mean().round(3))
display(check[["capital_gain", "log_capital_gain",
               "education_num", "hours_per_week", "edu_x_hours"]]
        .corrwith(check["over_50k"]).round(3))

age_group
25-34    0.168
35-44    0.330
45-54    0.403
55-64    0.323
65+      0.206
<25      0.012
Name: over_50k, dtype: float64

capital_gain        0.222
log_capital_gain    0.289
education_num       0.335
hours_per_week      0.228
edu_x_hours         0.370
dtype: float64

**Hypothesis check (train set only):**
- F2 supported: >50K rate is an inverted U by age – <25: 1.2%, 25-34: 16.8%, 35-44: 33.0%,
  **45-54: 40.3% (peak)**, 55-64: 32.3%, 65+: 20.6%.
- F1 supported: correlation with target rises from 0.222 (capital_gain) to 0.289 (log_capital_gain).
- F3 supported: edu_x_hours correlates 0.370, higher than education_num (0.335) or hours_per_week (0.228) alone.

## 5. Before/after comparison

In [21]:
engineered = make_model(X_train_fe).fit(X_train_fe, y_train)
engineered_scores = evaluate(engineered, X_test_fe, y_test)

comparison = pd.DataFrame([baseline_scores, engineered_scores],
                          index=["baseline", "engineered"])
comparison.loc["change"] = comparison.loc["engineered"] - comparison.loc["baseline"]
comparison.round(4)

,accuracy,f1,roc_auc
baseline,0.8548,0.6704,0.9083
engineered,0.8610,0.6857,0.9151
change,0.0063,0.0153,0.0068


In [22]:
from sklearn.model_selection import cross_val_score

cv_base = cross_val_score(make_model(X_train), X_train, y_train, cv=5, scoring="roc_auc")
cv_eng = cross_val_score(make_model(X_train_fe), X_train_fe, y_train, cv=5, scoring="roc_auc")
print("baseline:  ", cv_base.round(4), "mean", cv_base.mean().round(4))
print("engineered:", cv_eng.round(4), "mean", cv_eng.mean().round(4))
print("engineered better in", (cv_eng > cv_base).sum(), "of 5 folds")

baseline:   [0.9041 0.905  0.9134 0.8988 0.9002] mean 0.9043
engineered: [0.9091 0.9116 0.9177 0.9053 0.9083] mean 0.9104
engineered better in 5 of 5 folds


**Robustness:** 5-fold cross-validation on the training set (pipeline re-fit inside each fold)
gives mean ROC-AUC 0.9043 → 0.9104 (+0.0061). The engineered model wins in 5/5 folds
(gains +0.0043 to +0.0081), consistent with the held-out test gain of +0.0068.

## 6. Per-group check on the test set

In [23]:
from sklearn.metrics import recall_score

results = prot_test.assign(actual=y_test, predicted=engineered.predict(X_test_fe))
for col in ["sex", "race"]:
    summary = results.groupby(col).apply(lambda g: pd.Series({
        "n_test": len(g),
        "actual_rate": g["actual"].mean(),
        "predicted_rate": g["predicted"].mean(),
        "recall": recall_score(g["actual"], g["predicted"]),
    }))
    display(summary.round(3))

,n_test,actual_rate,predicted_rate,recall
sex,,,,
Female,2158.0,0.114,0.084,0.576
Male,4355.0,0.304,0.259,0.639


,n_test,actual_rate,predicted_rate,recall
race,,,,
Amer-Indian-Eskimo,73.0,0.123,0.055,0.333
Asian-Pac-Islander,200.0,0.260,0.250,0.577
Black,662.0,0.140,0.097,0.538
Other,45.0,0.089,0.089,1.000
White,5533.0,0.255,0.215,0.638


**Finding 5 – Removing `sex`/`race` did not remove group disparities.**
- Sex: recall 57.6% (Female) vs 63.9% (Male); women are predicted >50K at 8.4% vs an actual
  11.4% (74% of true rate) compared with 85% for men. Consistent with `relationship` acting
  as a proxy for sex (Finding 4).
- Race: recall 53.8% (Black, n=662) vs 63.8% (White, n=5,533) – a 10-point gap on a usable
  sample. Amer-Indian-Eskimo (n=73, 3 of 9 found) and Other (n=45, 4 of 4 found) are too
  small to draw conclusions; their 33% and 100% recall are noise (Finding 3).
- Implication: this model should not be used for decisions about individuals without a
  fairness mitigation step (e.g. per-group threshold review) and more data for small groups.

# I2 – Feature Engineering & Data Quality (UCI Adult Income)
**Task:** predict income >50K. Logistic regression; encoding/scaling inside a Pipeline fit on train only.

| | Accuracy | F1 | ROC-AUC |
|---|---|---|---|
| Baseline | 0.8548 | 0.6704 | 0.9083 |
| + 3 engineered features | 0.8610 | 0.6857 | 0.9151 |
| Change | +0.0063 | +0.0153 | +0.0068 |

5-fold CV confirms: 0.9043 → 0.9104, engineered better in 5/5 folds.

**Features (each with a hypothesis, verified on train):** `log_capital_gain`, `age_group`, `edu_x_hours`.
**Audit findings:** (1) hidden "?" missingness, not at random; (2) class imbalance 24/76;
(3) sex/race sampling skew and small groups; (4) redundant `education`, uninformative `fnlwgt`,
`relationship` as sex proxy; (5) recall gaps by sex and race persist after removing protected attributes.